## 🐈 Building the model (PyTorch variant)

In the journey of building a machine learning model, one of the first decisions is choosing the right type of model—predictive or generative. Predictive models focus on forecasting outcomes based on input data, while generative models aim to learn the underlying distribution of data to generate new samples.

Our usecase is categorized under predictive machine learning. There are many different ways to build a predictive model. For our use case, we chose neural networks. It has the ability to generalize better, handle complex patterns, and more expressive.

## 🐠 Install & Import packages

Again, we will need to install and import packages as we develop our notebook.

This will take a couple of minutes, and if `pip` gives an Error, don't worry about it. Things will just run fine regardless.

In [ ]:
%pip install -U pip
%pip -q install pyarrow scikit-learn onnx seaborn onnxruntime
%pip install -qU torch onnx onnxruntime
%pip list | awk '/torch|onnx/ {print $0}'

In [ ]:
# setup proxy settings to access the network (OPTIONAL)
proxies: dict = {
    "http": "",
    "https": "",
}

In [ ]:
import os, pickle, sys
from pathlib import Path
import logging, warnings
import random
from collections import OrderedDict

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# are we on a local laptop?
macos = False

# data libraries
import pandas as pd
import numpy as np

# sklearn
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# pytorch
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.cuda as tc
if macos is True:
    import torch.mps as tmps
from torch.nn import Module, Sequential, ReLU, Linear, Module, Softmax, CrossEntropyLoss
from torch.optim import Adam

# Set some seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# local libraries
parent_dir = os.path.abspath('..')
# the parent_dir could already be there if the kernel was not restarted, and we run this cell again
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

try:
    from libs.utilities import download_file
except ImportError as e:
    print(f"{e}")

# 📦 Load Data

We again load our two datasets, merge them, drop the NA columns just like before and select the input and output data.

Input data (X) is the feature matrix that contains the characteristics of each song.

Output data (y)is the target variable the model is trying to predict. In this case, y is the 'country' column which represents the country where the song is popular. The model will learn to predict the country based on the song features in X.

Before training the model, we need to prepare the data properly:  

1. Encoding the Target Variable (`y`):  Our target variable (`y`) is the **country** where a song is popular. However, machine learning models work with numbers, not text labels. We use a Label Encoder to convert country names into numerical values.  

2. Splitting the Data: We split the dataset into training, validation, and test sets to ensure the model generalizes well.  
   - Training set (`X_train, y_train`): Used to train the model.  
   - Validation set (`X_val, y_val`): Helps fine-tune the model and avoid overfitting.  
   - Test set (`X_test, y_test`): Used at the very end to evaluate how well the model performs on unseen data.  

1. Feature Scaling: Since our input features (e.g., `duration_ms`, `danceability`, `loudness`) have different ranges, we scale them between 0 and 1 using `MinMaxScaler`. This prevents large-valued features from dominating smaller ones and helps the model learn efficiently. 

In [ ]:
# data files to download
SONG_PROPERTIES: dict = {
    "file": "../dataset.local/parquet/song_properties.parquet",
    "url": 'https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_properties.parquet',
}

SONG_RANKINGS: dict = {
    "file": "../dataset.local/parquet/song_rankings.parquet",
    "url": 'https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_rankings.parquet',
}

# Download the datasets if they don't exist locally
download_file(SONG_PROPERTIES["url"], SONG_PROPERTIES["file"], proxy=proxies)
download_file(SONG_RANKINGS["url"], SONG_RANKINGS["file"], proxy=proxies)

In [ ]:
# torch custom dataset for this usecase
class JukeboxDataset(Dataset):
    def __init__(self, rankings_file, properties_file, features_columns, label_column=None):
        self.features_columns = features_columns
        self.label_column = label_column

        # read ranking data from parquet file
        self.rankings_file = pd.read_parquet(rankings_file)
        # remove missing values from the dataset (NaN)
        self.rankings_file.dropna()

        # read songs properties from parquet file
        self.properties_file = pd.read_parquet(properties_file)

        # join datasets and get relevant features inside
        self.merged_dataset = self.rankings_file.merge(self.properties_file, on='spotify_id', how='left')
        self.merged_dataset = self.merged_dataset[self.features_columns]

        # prepare labels if needed
        if self.label_column:
            self.labels_data = self.rankings_file[self.label_column]
            # encode labels
            self.label_encoder = LabelEncoder()
            encoded_labels = torch.from_numpy(self.label_encoder.fit_transform(self.labels_data))
            self.labels = F.one_hot(encoded_labels)

        # normalize dataset between [-1,1]
        # Scale the data to remove mean and have unit variance. The data will be between -1 and 1, which makes it a lot easier 
        # for the model to learn than random (and potentially large) values.
        # It is important to only fit the scaler to the training data, otherwise you are leaking information about the global 
        # distribution of variables (which is influenced by the test set) into the training set.
        self.minmaxscaler = MinMaxScaler((-1, 1))
        self.merged_dataset = pd.DataFrame(self.minmaxscaler.fit_transform(self.merged_dataset), index=self.merged_dataset.index, columns=self.merged_dataset.columns)

    def __len__(self):
        return len(self.merged_dataset)

    def __getitem__(self, idx):
        """
        Retrieves a single row from the dataset file.
        """
        dp = self.merged_dataset.iloc[idx]

        if self.label_column:
            label = self.labels[idx]
            return torch.tensor(dp.values).float(), label.float()
        else:
            return torch.tensor(dp.values).float()  # For unsupervised learning

In [ ]:
# features...
label_feature = 'country'
dataset_features = ['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# load datasets
song_dataset = JukeboxDataset(rankings_file=SONG_RANKINGS["file"],
                                 properties_file=SONG_PROPERTIES["file"],
                                 features_columns=dataset_features, label_column=label_feature)

# print dataset info
print(f"Dataset Contains {len(song_dataset)} items.")

In [ ]:
# get an item from the dataset
datapoint, label = song_dataset[0]
print(f"Datapoint: shape {datapoint.shape}, Label: shape {label.shape}")
print(f"Datapoint: type {type(datapoint)}, dtype {datapoint.dtype}, Label: type {type(label)} dtype {label.dtype}")

In [ ]:
# Split the data into training and testing sets so you have something to test the trained model with.
train_percentage = 0.8
training_data, test_data = random_split(song_dataset, [train_percentage, 1.0-train_percentage])

In [ ]:
# get or declare parameters
n_samples, n_feats = song_dataset.merged_dataset.shape
n_rows, n_classes = song_dataset.labels.shape
base_size = 32
learning_rate = 1e-3
batch_size = 128
epochs = 2

print(f"SUMMARY:\n Total Samples: {n_samples}, features: {n_feats}, classes: {n_classes}\n Training Samples: {len(training_data)}, Testing Samples: {len(test_data)}")

# detect device
device = "cpu"
if macos is True:
    if tmps.is_available():
        device = "mps"
else:
    if tc.is_available():
        device = "cuda"


print(f"DEVICE:\n Training on {device}")

# 🛟 Prepare to save the model

Before we start building neural network and training the model, let's prepare the environment to store the resulting artifacts. 

We need to store our model artifacts in an S3 buckets with folder called models/model-name/version/ for versioning reasons.

In [ ]:
# create local directories to save the model artifacts before starting building neural network and training the model
model_version = 1
artifact_path = f"models/jukebox/{model_version}/artifacts"
onnx_model_path = f"models/jukebox/{model_version}/onnx"
pt_model_path = f"models/jukebox/{model_version}/torch"

Path(artifact_path).mkdir(parents=True, exist_ok=True)
Path(onnx_model_path).mkdir(parents=True, exist_ok=True)
Path(pt_model_path).mkdir(parents=True, exist_ok=True)

with open(f"{artifact_path}/minmax_scaler.pkl", "wb") as handle:
    pickle.dump(song_dataset.minmaxscaler, handle)

with open(f"{artifact_path}/label_encoder.pkl", "wb") as handle:
    pickle.dump(song_dataset.label_encoder, handle)

song_dataset.merged_dataset.to_parquet(f"{artifact_path}/merged_dataset.parquet")
training_data.dataset.merged_dataset.to_parquet(f"{artifact_path}/train_dataset.parquet")
test_data.dataset.merged_dataset.to_parquet(f"{artifact_path}/test_dataset.parquet")

# 🚀 Build the model

The below piece of code is like creating a smart helper, aka Model, that learns to guess which countries might like a song based on its characteristics (its features). To process these features, our model will pass them through multiple layers of "neurons". These work much like in our brain and the more layers and neurons it has, the more capacity it has for learning.

At the end, our model uses what it learned to predict the countries that would enjoy the song the most. 

Finally, we check how well our model is doing at making these guesses!

In [ ]:
# build the neural network that will precict the country out of selected features
class CountryPredictorNetwork(Module):
    def __init__(self, n_inputs, hidden_len, n_outputs):
        super().__init__()
        self.predictor_network = Sequential(OrderedDict([
            ("input", Linear(n_inputs, hidden_len)),
            ("input_activation", ReLU()),
            ("linear_0", Linear(hidden_len, hidden_len * 2)),
            ("activation_0", ReLU()),
            ("linear_1", Linear(hidden_len * 2, hidden_len * 4)),
            ("activation_1", ReLU()),
            ("linear_2", Linear(hidden_len * 4, hidden_len * 8)),
            ("activation_2", ReLU()),
            ("output", Linear(hidden_len * 8, n_outputs)),
            #("output_activation", Softmax(dim=1)), # not needed if loss function is CrossEntropyLoss.
        ]))

    def forward(self, input_sample):
        return self.predictor_network(input_sample)

# 🪿 Model Summary
Now let's run `print(model)` which prints out the blueprint of your music recommendation helper. 

When you run it, you'll see a long table that shows all the different processing stages your song features go through before turning into country predictions (please don't be scared of the output length!).

The table shows each layer of your music brain 🧠, starting with how it receives the song characteristics, then how it processes them through various "thinking layers" (those `Linear` and `ReLU` parts), and finally how it produces its country predictions.

For each layer, you'll see its name, the shape of information flowing through it (like how many numbers are being processed), and how many "learning knobs" (`parameters`) that layer has to adjust as it gets better at predictions 🤓

At the bottom, you'll see the total number of these learning knobs which tells you how complex your recommendation system is. A higher number means your system can potentially learn more complex patterns in music preferences across different countries. (that's why you keep hearing number of parameters a lot in LLM conversations as well)

This summary can help you understand the complexity of your music recommendation system without needing to understand all the math happening behind the scenes.

In [ ]:
# instantiate a model object
model = CountryPredictorNetwork(n_feats, base_size, n_classes)

print(f"Model structure: {model}\n\n")
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} {param.dtype}\n")

# 🏃 Train the Model

Now we train our smart helper to predict which country might like a song based on its features. We set it to learn from the training data for 2 epochs, which means that it sees the full dataset two times. During each round, it looks at the song characteristics (scaled_x_train) and the country labels (y_train). It also predicts on a separate dataset called the validation dataset (X_val and y_val) after each epoch. This is to see how well it does on data it hasn't trained on yet.
Remember we split the data into three in an above cell. That was the reason :)

Once the training is finished, we print a message to let us know that our model is ready to make predictions!

In [ ]:
# prepare loss functions and optimizers
loss_fn = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=learning_rate)

# data loaders
training_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

NOTICE: It is possible to change number of epochs that you wish to run, raising the number will require more memory for each run, however it is possible to make the model dumber if you continue to raise the number of epochs, remember to restart kerbel after raising number of epochs.

In [ ]:
# model training function
def training_loop(model, dataloader, loss_function, optimizer, epoch, device="cpu"):
    b_size = len(dataloader.dataset)
    model.to(device)
    model.train()
    batch_loss = 0.0
    for b, (point, label) in enumerate(dataloader):
        # compute prediction
        pred = model(point.to(device))
        # measure loss
        loss = loss_function(pred.to(device), label.to(device))

        # backpropagate
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # calculate loss
        batch_loss += loss.item()

        if b % 1000 == 0:
            current_loss, batch_position = loss.item(), b * batch_size
            print(f" -> Running Loss: {current_loss:.4f}, Processed Samples: [{batch_position}/{b_size}]")

    # epoch update
    print(f"EPOCH {epoch}: Cumulative Loss: {batch_loss/b_size:.4f}")

# model test function
def test_loop(model, dataloader, device="cpu"):
    b_size = len(dataloader.dataset)
    model.to(device)
    model.eval()
    # loop over test data
    correct_guesses = 0
    total_guesses = 0
    with torch.no_grad():
        for b, (point, label) in enumerate(dataloader):
            # make predictions
            y_hat = model(point.to(device))
            # calculate softmax (probability distribution across classes)
            y_hat_prob = torch.softmax(y_hat, dim=0)
            _, category = torch.max(y_hat_prob, dim=0)

            # calculate accuracy
            total_guesses += label.size(0)
            correct_guesses += (category == label).sum().item()

    print(f"Model Accuracy: {100*correct_guesses/total_guesses:.2f}%")

In [ ]:
# run the training loop for some epochs
for i in range(epochs):
    print(f"============ EPOCH {i}/{epochs} ============")
    training_loop(model, training_dataloader, loss_fn, optimizer, i, device)

In [ ]:
# validate model
test_loop(model, test_dataloader, device)

# 🫡 Save the Model

Here we convert our trained song prediction model into a popular format called ONNX.  
We also save the original torch model for use later on so we easier can scan it.

In [ ]:
%pip install onnxscript
# export model to onnx
from torch import randn, save
import torch.onnx
input_signature = randn(1,n_feats).to(device)
onnx_model = torch.onnx.export(model, input_signature, dynamo=True)

# save ONNX checkpoint
onnx_model.save(f"{onnx_model_path}/country_predictor_model.onnx")

# also save model weights in torch format
save(model.state_dict(), f'{pt_model_path}/country_predictor_model.pt')

# 🔥 Load the Model for Testing

Here we load the model to predict which country might like a song. We open a session to the model and feed in the test data (X_test). The model outputs predictions, and identifies the countries.

The accuracy is calculated by comparing the predicted countries to the actual ones as we have the actual answers in our data. That's how we get the accuracy metrics.

We also create a confusion matrix to visualize the prediction results, using a heatmap to show how well the model's predictions match the actual labels. We want the predicted country to be the same as the actual country as much as possible, which is visualized by having dark squares on the diagonal, so that for example country 0 often is predicted as country 0 (not as any other country).
In other words, the darker the diagonal line, the closer we get to good predictions.

In [ ]:
# visualization
import seaborn as sns
from matplotlib import pyplot as plt
import onnxruntime as rt

In [ ]:
# load model for inference
onnx_inference_model = rt.InferenceSession(f"{onnx_model_path}/country_predictor_model.onnx", providers=rt.get_available_providers())

# get inputs and outputs of the onnx model
onnx_input = onnx_inference_model.get_inputs()[0]
onnx_output = onnx_inference_model.get_outputs()[0]
input_name = onnx_input.name
output_name = onnx_output.name

print(f"ONNX Model:\n Input: {input_name}, shape: {onnx_input.shape}\n Output: {output_name}, shape: {onnx_output.shape}")

In [ ]:
# make a prediction using the model
y_pred_temp = []
y_test_temp = []
for point, label in test_data:
    point = point.reshape(onnx_input.shape)
    y_pred_temp.append(onnx_inference_model.run([output_name], {input_name: point.numpy()}))
    label = label.reshape(onnx_output.shape)
    y_test_temp.append(label.numpy())

In [ ]:
# compute prediction max values
y_pred_argmax = [np.argmax(k[0]) for k in y_pred_temp]

# compute label max
y_test_argmax = [np.argmax(k) for k in y_test_temp]

In [ ]:
accuracy = np.sum([x == y for x,y in zip(y_pred_argmax, y_test_argmax)]) / len(y_pred_argmax)
print(f"ONNX Model Accuracy: {accuracy:.4f}")

c_matrix = confusion_matrix(y_test_argmax,y_pred_argmax)
ax = sns.heatmap(c_matrix, cmap='Blues')
ax.set_xlabel("Prediction")
ax.set_ylabel("Actual")
ax.set_title('Confusion Matrix')
plt.show()

And now we need to save the model in our S3 bucket to make it available outside of this Notebook. So please open up the [2-save_model.ipynb](2-save_model.ipynb) :)